In [1]:
import pandas as pd
import os
import requests
import json

In [2]:
base_dir = 'data/census'

In [14]:
dfs = []
for files in os.listdir(base_dir):
    if files.endswith('.xlsx') or files.endswith('.xls'):
        df = pd.read_excel(os.path.join(base_dir, files))
        year = int(files.split('_')[0][-4:])
        df['year'] = year
        print(year,df.shape)
        dfs.append(df)

df = pd.concat(dfs)
df = df.set_index(['year','HHNO','PPN']).reset_index()
print(df.head())
#dfs.to_csv('data/census/census_96_21.csv', index=False)

2011 (48498, 39)
2001 (43514, 36)
1996 (40423, 32)
2016 (49794, 46)
2021 (50625, 47)
2006 (45773, 36)
   year     HHNO  PPN  QRTYP  HHTYPE  TENURE  RENT  MORTHH  YRLEFT  UHSIZE  \
0  2011  4000011    0      3       1       2    99    99.0     9.0       3   
1  2011  4000011    1      3       1       2    99    99.0     9.0       3   
2  2011  4000011    2      3       1       2    99    99.0     9.0       3   
3  2011  4000011    3      3       1       2    99    99.0     9.0       3   
4  2011  4000063    0      3       1       4    99    99.0     9.0       2   

   ...  SFAREAGP1_MT  SFAREAGP2_MT  RCHI  RENG  ROTHERLANG  WCHI  WENG  \
0  ...           NaN           NaN   NaN   NaN         NaN   NaN   NaN   
1  ...           NaN           NaN   NaN   NaN         NaN   NaN   NaN   
2  ...           NaN           NaN   NaN   NaN         NaN   NaN   NaN   
3  ...           NaN           NaN   NaN   NaN         NaN   NaN   NaN   
4  ...           NaN           NaN   NaN   NaN         NaN 

In [20]:
ddict = pd.read_csv(os.path.join(base_dir,'data_dict.csv'))
ddict.head()

,no,field_name,description,field_width,years_available
0,1,HHNO,Household serial number,7,"2006, 2011, 2016"
1,2,PPN,Person serial number,2,"2006, 2011, 2016"
2,3,QRTYP,Type of quarters,1,"2006, 2011, 2016"
3,4,HHTYPE,Type of household,1,"2006, 2011, 2016"
4,5,TENURE,Tenure of accommodation,1,"2006, 2011, 2016"


In [24]:
completion = df.groupby('year').apply(
    lambda subset: subset.notnull().mean()
).T.merge(ddict[['field_name','description']],left_index=True,right_on='field_name',how='left')

In [26]:
completion[(completion.drop(columns=['field_name','description']) < 1).any(axis=1)]

,1996,2001,2006,2011,2016,2021,field_name,description
6.0,0.0,1.0,1.0,1.0,1.0,1.0,MORTHH,Monthly domestic household mortgage payment an...
7.0,0.0,1.0,1.0,1.0,1.0,1.0,YRLEFT,Outstanding period of mortgage payment or loan...
12.0,0.0,1.0,1.0,1.0,1.0,1.0,MORT_INC,Mortgage payment and loan repayment to income ...
21.0,0.0,1.0,1.0,1.0,1.0,1.0,ETHNIC,Ethnicity
42.0,0.0,0.0,0.0,1.0,1.0,1.0,HHCOMP11,Household composition (new classification)
NaN,0.0,0.0,0.0,1.0,1.0,0.0,INDUST_NEW,NaN
44.0,0.0,0.0,0.0,1.0,1.0,0.0,OCCUP_NEW,Occupation (ISCO-08)
45.0,1.0,1.0,1.0,1.0,0.0,1.0,INDUST,Industry (HSIC Version 1.1)
46.0,1.0,1.0,1.0,1.0,0.0,1.0,OCCUP,Occupation (ISCO-88)
33.0,0.0,0.0,0.0,0.0,1.0,1.0,SFAREAGP1_MT,Floor area of accommodation (in square metres)


In [30]:
completion[(completion.drop(columns=['field_name','description']) == 1).all(axis=1)]

,1996,2001,2006,2011,2016,2021,field_name,description
0.0,1.0,1.0,1.0,1.0,1.0,1.0,HHNO,Household serial number
1.0,1.0,1.0,1.0,1.0,1.0,1.0,PPN,Person serial number
2.0,1.0,1.0,1.0,1.0,1.0,1.0,QRTYP,Type of quarters
3.0,1.0,1.0,1.0,1.0,1.0,1.0,HHTYPE,Type of household
4.0,1.0,1.0,1.0,1.0,1.0,1.0,TENURE,Tenure of accommodation
5.0,1.0,1.0,1.0,1.0,1.0,1.0,RENT,Monthly domestic household rent
8.0,1.0,1.0,1.0,1.0,1.0,1.0,UHSIZE,Household size
9.0,1.0,1.0,1.0,1.0,1.0,1.0,DJHHINC,Monthly domestic household income
10.0,1.0,1.0,1.0,1.0,1.0,1.0,DJHHCOMP,Household composition (old classification)
11.0,1.0,1.0,1.0,1.0,1.0,1.0,RENT_INC,Rent to income ratio


In [32]:
df.to_csv(os.path.join(base_dir,'census_96_21.csv'),index=False)

In [ ]:
import requests
import json



url = "https://www.censtatd.gov.hk/api/post.php"

dfs = []
ids = ['140-09001','140-09002','140-09003','140-09004','140-09005']
years = ['1999','2004','2009','2014','2019']
for id,year in zip(ids,years):
  parameters ={
    "cv": {
      "GROUP": [
        "S1",
        "S2",
        "S3",
        "S4",
        "S5",
        "S6",
        "S7",
        "S8",
        "S9"
      ],
      "LQTYPE": [
        "1",
        "2",
        "3"
      ]
    },
    "sv": {
      "HOUSE_EXPD": [
        "Raw_hkd_d"
      ]
    },
    "period": {
      "start": year
    },
    "id": id,
    "lang": "en"
  }
  data = {'query': json.dumps(parameters)}
  r = requests.post(url, data=data, timeout=20)
  print(year,r.status_code)
  try:
    data = r.json()['dataSet']
    df = pd.DataFrame(data)
    print(df.describe())
    dfs.append(df)
  except:
    print(r.text)
    break




1999 200
             figure
count     40.000000
mean    4202.250000
std     6352.400522
min      164.000000
25%      724.250000
50%     1251.000000
75%     5186.000000
max    27239.000000
2004 200
             figure
count     40.000000
mean    3537.150000
std     5364.973886
min      138.000000
25%      683.000000
50%     1134.000000
75%     4242.000000
max    23920.000000
2009 200
            figure
count     40.00000
mean    4009.75000
std     6284.33141
min      117.00000
25%      644.00000
50%     1154.00000
75%     5052.75000
max    28715.00000
2014 200
             figure
count     40.000000
mean    5185.275000
std     8218.943809
min      131.000000
25%      750.500000
50%     1270.000000
75%     6507.750000
max    36728.000000
2019 200
             figure
count     40.000000
mean    5538.675000
std     8775.684534
min      126.000000
25%      746.500000
50%     1448.500000
75%     7265.250000
max    37895.000000


In [29]:
big_df = pd.concat(dfs)
big_df = big_df[big_df['GROUPDesc']!='Total']
#big_df = big_df[big_df['LQTYPEDesc']!='Total']

In [30]:
big_df.head()

,GROUP,GROUPDesc,LQTYPE,LQTYPEDesc,freq,period,sv,svDesc,figure,sd_value
4,S1,Food,,Total,Y,1999,HOUSE_EXPD,HK$,5612,
5,S1,Food,1,Public housing,Y,1999,HOUSE_EXPD,HK$,5044,
6,S1,Food,2,Subsidised housing,Y,1999,HOUSE_EXPD,HK$,5772,
7,S1,Food,3,Private housing,Y,1999,HOUSE_EXPD,HK$,5910,
8,S2,Housing,,Total,Y,1999,HOUSE_EXPD,HK$,7009,


In [31]:
big_df.to_csv('data/household/household_expenditure.csv',index=False)

In [12]:
import requests
import json



url = "https://www.censtatd.gov.hk/api/post.php"
parameters ={
  "cv": {
    "LQTYPE": [
      "1",
      "2",
      "3",
      "4"
    ]
  },
  "sv": {
    "DH": [
      "Raw_K_1dp_hh_n"
    ]
  },
  "period": {
    "start": "2002"
  },
  "id": "130-06603",
  "lang": "en"
}
data = {'query': json.dumps(parameters)}
r = requests.post(url, data=data, timeout=20)
try:
  data = r.json()['dataSet']
  df = pd.DataFrame(data)
  print(df.describe())
  print(df.head())

  #df = df[df['GROUPDesc']!='Total']
  df = df[df['LQTYPEDesc']!='Total']
  df.to_csv('data/household/household_types.csv',index=False)

except Exception as e:
  print(e)
  print(r.text)

            figure
count   120.000000
mean    973.141667
std     852.947156
min      15.000000
25%     370.875000
50%     737.650000
75%    1406.325000
max    2778.100000
  LQTYPE                         LQTYPEDesc freq period  sv      svDesc  \
0                                     Total    Y   2002  DH  No. ('000)   
1      1              Public rental housing    Y   2002  DH  No. ('000)   
2      2  Subsidised home ownership housing    Y   2002  DH  No. ('000)   
3      3          Private permanent housing    Y   2002  DH  No. ('000)   
4      4                  Temporary housing    Y   2002  DH  No. ('000)   

   figure sd_value  
0  2080.5           
1   629.5           
2   355.3           
3  1069.5           
4    26.2           


,LQTYPE,LQTYPEDesc,freq,period,sv,svDesc,figure,sd_value
0,,Total,Y,2002,DH,No. ('000),2080.5,
1,1,Public rental housing,Y,2002,DH,No. ('000),629.5,
2,2,Subsidised home ownership housing,Y,2002,DH,No. ('000),355.3,
3,3,Private permanent housing,Y,2002,DH,No. ('000),1069.5,
4,4,Temporary housing,Y,2002,DH,No. ('000),26.2,
...,...,...,...,...,...,...,...,...
115,,Total,Y,2025,DH,No. ('000),2778.1,
116,1,Public rental housing,Y,2025,DH,No. ('000),834.8,
117,2,Subsidised home ownership housing,Y,2025,DH,No. ('000),426.3,
118,3,Private permanent housing,Y,2025,DH,No. ('000),1489.9,


In [8]:
df.to_csv('data/household/household_types.csv',index=False)

In [23]:

url = "https://www.censtatd.gov.hk/api/post.php"
parameters ={
  "cv": {
    "LQTYPE": [
      "1",
      "2",
      "3"
    ]
  },
  "sv": {
    "ADHS": [
      "Raw_1dp_per_n"
    ]
  },
  "period": {
    "start": "2002"
  },
  "id": "130-06602",
  "lang": "en"
}
data = {'query': json.dumps(parameters)}
r = requests.post(url, data=data, timeout=20)
try:
  data = r.json()['dataSet']
  df = pd.DataFrame(data)
  print(df.describe())
  print(df.head())

  #df = df[df['GROUPDesc']!='Total']
  #df = df[df['LQTYPEDesc']!='Total']

  print(df.head())
  df.to_csv('data/household/household_sizes.csv',index=False)
except Exception as e:
  print(e)
  print(r.text)

          figure
count  96.000000
mean    2.909375
std     0.214269
min     2.600000
25%     2.700000
50%     2.900000
75%     3.000000
max     3.500000
  LQTYPE                         LQTYPEDesc freq period    sv svDesc  figure  \
0                                     Total    Y   2002  ADHS    No.     3.2   
1      1              Public rental housing    Y   2002  ADHS    No.     3.3   
2      2  Subsidised home ownership housing    Y   2002  ADHS    No.     3.5   
3      3          Private permanent housing    Y   2002  ADHS    No.     3.0   
4                                     Total    Y   2003  ADHS    No.     3.1   

  sd_value  
0           
1           
2           
3           
4           
  LQTYPE                         LQTYPEDesc freq period    sv svDesc  figure  \
0                                     Total    Y   2002  ADHS    No.     3.2   
1      1              Public rental housing    Y   2002  ADHS    No.     3.3   
2      2  Subsidised home ownership housing    Y

In [16]:
url = "https://www.censtatd.gov.hk/api/post.php"
parameters ={
  "cv": {
    "LQTYPE": [
      "1",
      "2",
      "3_temp"
    ]
  },
  "sv": {
    "MED_DH_INC": [
      "Raw_hkd_d"
    ],
    "MED_DH_INC_XEI": [
      "Raw_hkd_d"
    ]
  },
  "period": {
    "start": "2002"
  },
  "id": "130-06610",
  "lang": "en"
}
data = {'query': json.dumps(parameters)}
r = requests.post(url, data=data, timeout=20)
try:
  data = r.json()['dataSet']
  df = pd.DataFrame(data)
  print(df.describe())
  print(df.head())

  #df = df[df['GROUPDesc']!='Total']
  df = df[df['LQTYPEDesc']!='Total']

  print(df.head())
  df.to_csv('data/household/household_income.csv',index=False)
except Exception as e:
  print(e)
  print(r.text)

             figure
count    192.000000
mean   25597.916667
std     9226.021465
min    10500.000000
25%    19350.000000
50%    24900.000000
75%    30575.000000
max    52400.000000
   LQTYPE                         LQTYPEDesc freq period          sv svDesc  \
0                                      Total    Y   2002  MED_DH_INC    HK$   
1       1              Public rental housing    Y   2002  MED_DH_INC    HK$   
2       2  Subsidised home ownership housing    Y   2002  MED_DH_INC    HK$   
3  3_temp          Private permanent housing    Y   2002  MED_DH_INC    HK$   
4                                      Total    Y   2003  MED_DH_INC    HK$   

   figure sd_value  
0   17000           
1   11000           
2   20000           
3   22500           
4   16000           
   LQTYPE                         LQTYPEDesc freq period          sv svDesc  \
1       1              Public rental housing    Y   2002  MED_DH_INC    HK$   
2       2  Subsidised home ownership housing    Y   2002  MED

In [18]:
url = "https://www.censtatd.gov.hk/api/post.php"
parameters ={
  "cv": {
    "GROUP": [
      "S1",
      "S2",
      "S3",
      "S4",
      "S5",
      "S6",
      "S7",
      "S8",
      "S9"
    ]
  },
  "sv": {
    "CC_CM_1920": [
      "Raw_1dp_idx_n"
    ],
    "B_CM_1920": [
      "Raw_1dp_idx_n"
    ],
    "C_CM_1920": [
      "Raw_1dp_idx_n"
    ],
    "A_CM_1920": [
      "Raw_1dp_idx_n"
    ]
  },
  "period": {
    "start": "1999"
  },
  "id": "510-60001A",
  "lang": "en"
}
data = {'query': json.dumps(parameters)}
r = requests.post(url, data=data, timeout=20)
try:
  data = r.json()['dataSet']
  df = pd.DataFrame(data)
  print(df.describe())
  print(df.head())

  #df = df[df['GROUPDesc']!='Total']
  #df = df[df['LQTYPEDesc']!='Total']
  

  print(df.head())
  df.to_csv('data/household/consumer_price.csv',index=False)
except Exception as e:
  print(e)
  print(r.text)

            figure
count  1080.000000
mean     93.480370
std      26.297659
min      51.000000
25%      77.200000
50%      93.150000
75%     102.100000
max     248.600000
  GROUP                     GROUPDesc freq period          sv svDesc  figure  \
0                               Total    Y   1999  CC_CM_1920  Index    73.6   
1    S1                          Food    Y   1999  CC_CM_1920  Index    56.9   
2    S2                       Housing    Y   1999  CC_CM_1920  Index    75.2   
3    S3    Electricity, gas and water    Y   1999  CC_CM_1920  Index    89.0   
4    S4  Alcoholic drinks and tobacco    Y   1999  CC_CM_1920  Index    58.6   

  sd_value  
0           
1           
2           
3           
4           
  GROUP                     GROUPDesc freq period          sv svDesc  figure  \
0                               Total    Y   1999  CC_CM_1920  Index    73.6   
1    S1                          Food    Y   1999  CC_CM_1920  Index    56.9   
2    S2                       

In [7]:
url = "https://www.censtatd.gov.hk/api/post.php"
parameters ={

  "cv": {
    "GDP_COMPONENT": [
      "PCE"
    ]
  },
  "sv": {
    "CUR": [
      "Ratio_1dp_%_n"
    ]
  },
  "period": {
    "start": "1999"
  },
  "id": "310-31006",
  "lang": "en"
}
data = {'query': json.dumps(parameters)}
r = requests.post(url, data=data, timeout=20)
try:
  data = r.json()['dataSet']
  df = pd.DataFrame(data)
  print(df.describe())
  print(df.head())

  df.to_csv('data/household/gdp_pce.csv',index=False)
except Exception as e:
  print(e)

  print(r.text)

           figure
count   54.000000
mean    81.648148
std     18.742614
min     57.500000
25%     64.700000
50%     85.000000
75%    100.000000
max    100.000000
  GDP_COMPONENT                GDP_COMPONENTDesc freq period   sv svDesc  \
0                                          Total    Y   1999  CUR      %   
1           PCE  Private consumption expenditure    Y   1999  CUR      %   
2                                          Total    Y   2000  CUR      %   
3           PCE  Private consumption expenditure    Y   2000  CUR      %   
4                                          Total    Y   2001  CUR      %   

   figure sd_value  
0   100.0           
1    60.2           
2   100.0           
3    58.6           
4   100.0           


In [9]:
url = "https://www.censtatd.gov.hk/api/post.php"
parameters ={
  "cv": {
    "GDP_COMPONENT": [
      "PCE"
    ]
  },
  "sv": {
    "CUR": [
      "Raw_M_hkd_d"
    ]
  },
  "period": {
    "start": "1999"
  },
  "id": "310-31002",
  "lang": "en"
}
data = {'query': json.dumps(parameters)}
r = requests.post(url, data=data, timeout=20)
try:
  data = r.json()['dataSet']
  df = pd.DataFrame(data)
  print(df.describe())
  print(df.head())

  df.to_csv('data/household/gdp_pce.csv',index=False)
except Exception as e:
  print(e)
  print(r.text)

             figure
count  5.400000e+01
mean   1.735114e+06
std    7.034833e+05
min    7.229610e+05
25%    1.263988e+06
50%    1.655000e+06
75%    2.142956e+06
max    3.331774e+06
  GDP_COMPONENT                GDP_COMPONENTDesc freq period   sv  \
0                                          Total    Y   1999  CUR   
1           PCE  Private consumption expenditure    Y   1999  CUR   
2                                          Total    Y   2000  CUR   
3           PCE  Private consumption expenditure    Y   2000  CUR   
4                                          Total    Y   2001  CUR   

        svDesc   figure sd_value  
0  HK$ million  1285946           
1  HK$ million   774701           
2  HK$ million  1337501           
3  HK$ million   784323           
4  HK$ million  1321142           


In [3]:
import requests
import json
url = "https://www.censtatd.gov.hk/api/post.php"
dfs = []
ids = ['140-09001','140-09002','140-09003','140-09004','140-09005']
years = ['1999','2004','2009','2014','2019']
for id,year in zip(ids,years):
  parameters ={
    "cv": {
      "GROUP": [
        "S2",
        "S6",
        "S9",
        "2009_28",
        "2009_29",
        "2009_30",
        "2009_51",
        "2009_52",
        "2009_53",
        "2009_54",
        "2009_55",
        "2009_56",
        "2009_57",
        "2009_58",
        "2009_81",
        "2009_82",
        "2009_83",
        "2009_84",
        "2009_85",
        "2009_86",
        "2009_87",
        "2009_88",
        "2009_89",
        "2009_90",
        "2009_91",
        "2009_92",
        "2009_93",
        "2009_94"
      ],
      "LQTYPE": [
        "1",
        "2",
        "3"
      ]
    },
    "sv": {
      "HOUSE_EXPD": [
        "Raw_hkd_d"
      ]
    },
    "period": {
      "start": "2009"
    },
    "id": "140-09003",
    "lang": "en"
  }
  data = {'query': json.dumps(parameters)}
  r = requests.post(url, data=data, timeout=20)
  print(year,r.status_code)
  try:
    data = r.json()['dataSet']
    df = pd.DataFrame(data)
    print(df.describe())
    dfs.append(df)
  except:
    print(r.text)
    break

1999 200
       GROUP GROUPDesc LQTYPE LQTYPEDesc freq period          sv svDesc  \
count    116       116    116        116  116    116         116    116   
unique    29        29      4          4    1      1           1      1   
top              Total             Total    Y   2009  HOUSE_EXPD    HK$   
freq       4         4     29         29  116    116         116    116   

        figure sd_value  
count      116      116  
unique      96        2  
top         31           
freq         4      115  
2004 200
       GROUP GROUPDesc LQTYPE LQTYPEDesc freq period          sv svDesc  \
count    116       116    116        116  116    116         116    116   
unique    29        29      4          4    1      1           1      1   
top              Total             Total    Y   2009  HOUSE_EXPD    HK$   
freq       4         4     29         29  116    116         116    116   

        figure sd_value  
count      116      116  
unique      96        2  
top         31        

In [4]:
big_df = pd.concat(dfs)

In [6]:
big_df = pd.concat(dfs)
big_df = big_df[big_df['GROUPDesc']!='Total']
#big_df = big_df[big_df['LQTYPEDesc']!='Total']

In [7]:
big_df.to_csv('data/household/household_expenditure_granular.csv',index=False)

In [100]:
df = pd.read_excel('data/household/jp19.xls')

In [101]:
df.columns = df.loc[13]
df = df.loc[62:180]
df = df.loc[df.iloc[:,10].notna()]
df.index = df.iloc[:,-4]
df = df.iloc[:,16:18]
df.columns = ['expense','share']
df = df.reset_index()
df.columns = ['goods','expense','share']

In [102]:
print(df.sum())
df

goods      FoodHousingFuel, light & water chargesFurnitur...
expense                                               253720
share                                                  100.2
dtype: object


,goods,expense,share
0,Food,59258,23.4
1,Housing,18402,7.3
2,"Fuel, light & water charges",18435,7.3
3,Furniture & household utensils,8448,3.3
4,Clothing & footwear,10572,4.2
5,Medical care,10891,4.3
6,Transportation & communication,32910,13
7,Education,9112,3.6
8,Culture & recreation,28396,11.2
9,Other consumption expenditures,57296,22.6


In [103]:
df.to_csv('data/household/jp19.csv',index=False)

In [106]:
names = ['jp04','jp09','jp14','jp19']
years = [2004,2009,2014,2019]
dfs = []
for name,year in zip(names,years):
    df = pd.read_csv(f'data/household/{name}.csv')
    df['period'] = year
    dfs.append(df)

df = pd.concat(dfs)
df
df.to_csv('data/household/jp.csv',index=False)


In [116]:
df = pd.read_csv('data/household/sg.csv')

In [117]:
df

,Year,2023,Unnamed: 2,2017/18,Unnamed: 4,2012/13,Unnamed: 6,2007/08,Unnamed: 8,2002/03,Unnamed: 10,Unnamed: 11
0,Type of Goods and Services (2-digit),Distribution of Household Expenditure (%),Average Monthly Household Expenditure ($),Distribution of Household Expenditure (%),Average Monthly Household Expenditure ($),Distribution of Household Expenditure (%),Average Monthly Household Expenditure ($),Distribution of Household Expenditure (%),Average Monthly Household Expenditure ($),Distribution of Household Expenditure (%),Average Monthly Household Expenditure ($),NaN
1,"TOTAL, including imputed rental for owner-occu...",100,7118.7,100,6160.8,100,5815.3,100,4432.5,100,3797.8,NaN
2,FOOD AND NON-ALCOHOLIC BEVERAGES,6.4,455.9,6.4,394,7.5,433.3,8.2,364.6,9,340.5,NaN
3,ALCOHOLIC BEVERAGES AND TOBACCO,0.6,39.7,0.7,42.4,0.9,53.4,1,46.5,1.4,52.2,NaN
4,CLOTHING AND FOOTWEAR,1.7,119.6,2,122.9,2.7,155.6,3.2,143.1,3.3,127,NaN
5,HOUSING AND UTILITIES,24.4,1737.5,23.1,1425.6,25.3,1473.6,22.2,983.6,18.5,704.1,NaN
6,Imputed Rental for Owner-Occupied Accommodation,16.7,1187.7,16.2,998,18,1046.8,14.1,623.6,11.7,445.9,NaN
7,"FURNISHING, HOUSEHOLD EQUIPMENT AND ROUTINE HO...",5.4,384.6,4.5,277.5,4.4,256.7,4.1,182.1,5.6,212.5,NaN
8,HEALTH,6.7,473.5,5.2,320.1,4.3,253,4.8,211.4,4.3,163.5,NaN
9,TRANSPORT,13.4,951.2,16.9,1038.2,14.7,856.7,15.8,701.9,16.3,618.6,NaN


In [ ]:
df = df.set_index('Year')
df = df.iloc[:,[1,3,5,7,9]]
df.columns = [2022,2017,2012,2007,2002]

In [ ]:
df.columns = [2022,2017,2012,2007,2002]

,Unnamed: 2,Unnamed: 4,Unnamed: 6,Unnamed: 8,Unnamed: 10
Year,,,,,
Type of Goods and Services (2-digit),Average Monthly Household Expenditure ($),Average Monthly Household Expenditure ($),Average Monthly Household Expenditure ($),Average Monthly Household Expenditure ($),Average Monthly Household Expenditure ($)
"TOTAL, including imputed rental for owner-occupied accommodation",7118.7,6160.8,5815.3,4432.5,3797.8
FOOD AND NON-ALCOHOLIC BEVERAGES,455.9,394,433.3,364.6,340.5
ALCOHOLIC BEVERAGES AND TOBACCO,39.7,42.4,53.4,46.5,52.2
CLOTHING AND FOOTWEAR,119.6,122.9,155.6,143.1,127
HOUSING AND UTILITIES,1737.5,1425.6,1473.6,983.6,704.1
Imputed Rental for Owner-Occupied Accommodation,1187.7,998,1046.8,623.6,445.9
"FURNISHING, HOUSEHOLD EQUIPMENT AND ROUTINE HOUSEHOLD MAINTENANCE",384.6,277.5,256.7,182.1,212.5
HEALTH,473.5,320.1,253,211.4,163.5
